<div style="
 background: linear-gradient(145deg, #1a0b08, #2d1310);
 border: 4px solid transparent;
 border-radius: 14px;
 padding: 18px 22px;
 margin: 12px 0;
 font-size: 26px;
 font-weight: 600;
 color: #fff8f6;
 box-shadow: 0 6px 14px rgba(0,0,0,0.3);
 background-clip: padding-box;
 position: relative;
">
 <div style="
 position: absolute;
 inset: 0;
 padding: 4px;
 border-radius: 14px;
 background: linear-gradient(90deg, #ff7b00, #ff0054, #9d0208);
 -webkit-mask: 
 linear-gradient(#fff 0 0) content-box, 
 linear-gradient(#fff 0 0);
 -webkit-mask-composite: xor;
 mask-composite: exclude;
 pointer-events: none;
 "></div>
 
 <b>10 $\rightarrow$ LangGraph Chatbot</b> 
 <span style="color:#ffb5a7;">(Stateful AI Agents in LangGraph)</span>
</div>

## 📑 **Table of Contents**

- **1.** [Introduction](#1-introduction)
- **2.** [Chatbot Architecture](#2-chatbot-architecture)
- **3.** [State Management](#3-state-management)
  - **3.1** [Defining the State](#31-defining-the-state)
  - **3.2** [Message Types](#32-message-types)
  - **3.3** [Reducer Functions](#33-reducer-functions)
- **4.** [Building the Graph](#4-building-the-graph)
  - **4.1** [The Chat Node](#41-the-chat-node)
  - **4.2** [Defining Edges](#42-defining-edges)
- **5.** [Implementing the Chat Loop](#5-implementing-the-chat-loop)
- **6.** [Short-Term Memory](#6-short-term-memory)
  - **6.1** [The Memory Problem](#61-the-memory-problem)
  - **6.2** [Implementing Persistence](#62-implementing-persistence)
  - **6.3** [Threads and Configurations](#63-threads-and-configurations)

---


## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">🎯 Learning Objectives</span>

After completing this notebook, you will be able to:

* ✅ Design a **sequential workflow** for an LLM-based chatbot using LangGraph.
* ✅ Define and manage complex **state structures** holding conversational history.
* ✅ Utilize standard message abstractions (`BaseMessage`, `HumanMessage`, `AIMessage`).
* ✅ Implement **reducer functions** (`add_messages`) to safely append state data without overwriting.
* ✅ Build an **interactive, continuous chat loop**.
* ✅ Diagnose **state-loss issues** in iterative invocations.
* ✅ Implement **short-term memory** using Checkpointers and Thread IDs.

---


## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">📋 Prerequisites</span>

* 🐍 Proficiency in **Python** programming.
* 🧩 Understanding of basic **LangGraph** concepts (Nodes, Edges, State, and Graph compilation).
* 🤖 Familiarity with **Large Language Models (LLMs)** and standard messaging formats.

---


In [1]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

from dotenv import load_dotenv
load_dotenv()

True

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">1. Introduction</span>


A robust chatbot requires more than simply passing a string to an LLM. It demands structured **state management** to remember past interactions, integration pathways for advanced tools (like Retrieval-Augmented Generation or API calls), and a resilient architecture.

This guide covers the foundational step: building a **continuous conversational agent** equipped with short-term memory using LangGraph.

---


## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">2. Chatbot Architecture</span>


The core architecture of a basic chatbot is a simple **sequential workflow** consisting of a single node.

**Execution Flow:**

1. **Start:** The user's input message is received.
2. **Chat Node:** The message (along with past conversation history) is routed to an LLM.
3. **Response Generation:** The LLM processes the history and generates a response.
4. **End:** The response is returned to the user, concluding that specific execution iteration.

This workflow is repeatedly triggered in a loop to create a continuous chat experience.

---


## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">3. State Management</span>


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 7px 16px; border-radius: 8px; font-size: 17px; font-weight: 700;">3.1 Defining the State</span>

In LangGraph, the `State` dictionary holds the critical data required during workflow execution. For a chatbot, the most crucial data is the conversation history — the chronological exchange of messages between the user and the AI.

The state is defined as a `TypedDict` containing a single attribute: a **list of messages**.

---

### 3.2 Message Types

Instead of storing raw strings, it is best practice to store standardized **message objects**. These objects inherit from `BaseMessage` and clarify the origin and intent of the text:

| Type | Description |
| --- | --- |
| `HumanMessage` | Input provided by the user |
| `AIMessage` | Responses generated by the LLM |
| `SystemMessage` | System-level instructions dictating the LLM's persona or behavior |
| `ToolMessage` | Outputs generated by external tools |

Defining the state list to accept `BaseMessage` ensures flexibility to handle all these types seamlessly.

---

### 3.3 Reducer Functions

By default, updating a key in a LangGraph state dictionary **overwrites** the existing value. For example, passing a new `AIMessage` would normally delete the preceding `HumanMessage`.

To maintain the conversation history, a **Reducer Function** must be applied to the state attribute. While a standard operator (like `operator.add`) can concatenate lists, LangGraph provides a specialized reducer named `add_messages`.

**Purpose of `add_messages`:** It is highly optimized for handling `BaseMessage` objects, ensuring that new messages are **appended** to the existing list rather than replacing it.

<div style="background: linear-gradient(135deg, #2d1b00, #3d2200); border-left: 4px solid #f4a261; padding: 14px 18px; border-radius: 8px; margin: 10px 0; color: #eee; font-size: 14px;">⚠️ <b>Warning:</b> Without a reducer, every new invocation would overwrite the full message history, causing the chatbot to forget all previous context.</div>

#### Syntax: State Definition

```python
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    # Annotated binds the data type with the reducer function
    messages: Annotated[list[BaseMessage], add_messages]
```

---


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">4. Building the Graph</span>


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 7px 16px; border-radius: 8px; font-size: 17px; font-weight: 700;">4.1 The Chat Node</span>

The chat node is a Python function that performs the following:

1. Instantiates the LLM.
2. Extracts the current conversation history (`messages`) from the State.
3. Invokes the LLM with the history.
4. Returns the generated response as a list — which the reducer will **append** to the main state.

<div style="background: linear-gradient(135deg, #0d1b2a, #1b263b); border-left: 4px solid #52b788; padding: 14px 18px; border-radius: 8px; margin: 10px 0; color: #eee; font-size: 14px;">💡 <b>Tip:</b> Returning the response wrapped in a list (`[response]`) ensures the `add_messages` reducer appends rather than overwrites.</div>

#### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 5px 12px; border-radius: 6px; font-size: 14px; font-weight: 600;">📝 Code Example: The Chat Node</span>



In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant")

def chat_node(state: ChatState):
    # Extract messages from state
    current_messages = state["messages"]
    # Invoke LLM with full conversation history
    response = llm.invoke(current_messages)
    # Return as list; add_messages reducer appends it to history
    return {"messages": [response]}

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 7px 16px; border-radius: 8px; font-size: 17px; font-weight: 700;">4.2 Defining Edges</span>

The workflow is constructed by tying the node to the graph's entry and exit points.

* **`StateGraph(ChatState)`** — Initializes the workflow blueprint tied to `ChatState`.
* **`START`** — Entry point that triggers the `chat_node` with the user's input.
* **`END`** — Exit point after the LLM generates a response.

#### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 5px 12px; border-radius: 6px; font-size: 14px; font-weight: 600;">📝 Code Example: Graph Compilation</span>



In [ ]:
# Initialize the graph with the defined state
builder = StateGraph(ChatState)

# Add the node
builder.add_node("chat_node", chat_node)

# Define the execution edges
builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", END)

# Compile the workflow (without memory for now)
chatbot = builder.compile()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">5. Implementing the Chat Loop</span>


To simulate an interactive chatbot environment, the compiled graph must be invoked iteratively via an infinite `while` loop. The loop captures user input, packages it into a `HumanMessage`, invokes the workflow, and extracts the AI's response.

<div style="background: linear-gradient(135deg, #2d1b00, #3d2200); border-left: 4px solid #f4a261; padding: 14px 18px; border-radius: 8px; margin: 10px 0; color: #eee; font-size: 14px;">⚠️ <b>Warning:</b> At this stage, the chatbot has <b>no persistent memory</b>. Each invocation starts fresh — the state is erased when the graph reaches <code>END</code>.</div>

#### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 5px 12px; border-radius: 6px; font-size: 14px; font-weight: 600;">📝 Code Example: Interactive Loop</span>



In [ ]:
# Uncomment to run interactively in a terminal
while True:
    user_input = input("Type here: ")
    clean_input = user_input.strip().lower()
    if clean_input in ["exit", "quit", "bye"]:
        print("Ending chat. Goodbye!")
        break
    initial_state = {"messages": [HumanMessage(content=user_input)]}
    print(f"User: {user_input}")
    response_state = chatbot.invoke(initial_state)
    ai_message = response_state["messages"][-1].content
    print(f"AI: {ai_message}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #065f46, #854d0e); padding: 12px 20px; border-radius: 12px; font-size: 24px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2); transition: transform 0.2s ease, box-shadow 0.2s ease;">6. Short-Term Memory</span>


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 7px 16px; border-radius: 8px; font-size: 17px; font-weight: 700;">6.1 The Memory Problem</span>

If the above chat loop is executed, the chatbot will suffer from **amnesia**. It will successfully answer isolated questions but will fail to recall information established in previous prompts.

**Example:**
- User: *"My name is John"* → AI answers.
- User: *"What is my name?"* → AI **cannot recall** — it never saw the first message.

**The Cause:** Every time `chatbot.invoke(initial_state)` is called within the loop, the workflow executes from `START` to `END`. Upon reaching `END`, the in-memory state for that specific execution is **flushed**. The next iteration passes only the newest `HumanMessage`.

<div style="background: linear-gradient(135deg, #2d1b00, #3d2200); border-left: 4px solid #f4a261; padding: 14px 18px; border-radius: 8px; margin: 10px 0; color: #eee; font-size: 14px;">⚠️ <b>Warning:</b> The LLM never sees preceding conversation turns, because the state is wiped on every invocation.</div>

---


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 7px 16px; border-radius: 8px; font-size: 17px; font-weight: 700;">6.2 Implementing Persistence</span>

To solve the memory problem, **Persistence** must be implemented. Persistence ensures that when the workflow reaches `END`, the accumulated state is **saved to a Checkpointer**.

During the next invocation, the workflow:
1. Retrieves the saved state from the Checkpointer.
2. Appends the new message using the `add_messages` reducer.
3. Passes the full history to the LLM.

<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); border-left: 4px solid #e94560; padding: 14px 18px; border-radius: 8px; margin: 10px 0; color: #eee; font-size: 14px;">📌 <b>Note:</b> For development, `MemorySaver` (in-memory RAM) is sufficient. Production environments require database-backed checkpointers like <b>SQLite</b> or <b>PostgreSQL</b> to persist data between server restarts.</div>

---


### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 7px 16px; border-radius: 8px; font-size: 17px; font-weight: 700;">6.3 Threads and Configurations</span>

When persistence is enabled, the checkpointer needs a way to identify which conversation to retrieve. This is handled via a **Thread ID**.

Every unique conversation session must be assigned a unique Thread ID, passed via a configuration dictionary during invocation.

<div style="background: linear-gradient(135deg, #0d1b2a, #1b263b); border-left: 4px solid #52b788; padding: 14px 18px; border-radius: 8px; margin: 10px 0; color: #eee; font-size: 14px;">💡 <b>Tip:</b> Think of a Thread ID like a unique session cookie — it lets the checkpointer isolate and retrieve the correct conversation history for each user.</div>

#### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #1e3a5f, #1565c0); padding: 5px 12px; border-radius: 6px; font-size: 14px; font-weight: 600;">📝 Code Example: Chatbot with Memory</span>


#### 🔧 Internal Working of Persistence:

1. User sends *"My name is John"*.
2. State is retrieved (empty). `HumanMessage` is added. LLM replies with `AIMessage`.
3. Execution ends. The Checkpointer **saves** the state (User msg + AI msg) against `thread_id: session_001` in RAM.
4. User asks *"What is my name?"*.
5. `invoke` is called with `config={"configurable": {"thread_id": "session_001"}}`.
6. LangGraph **fetches** the saved state from RAM.
7. The new message is appended (using `add_messages`).
8. The LLM receives the **full history** and answers correctly.
9. Checkpointer **updates** the saved state.

---


In [ ]:
# 1. Re-compile the graph with a Checkpointer
memory = MemorySaver()
chatbot_with_memory = builder.compile(checkpointer=memory)

# 2. Define the execution Configuration (Thread ID)
# This identifies a specific chat session.
config = {"configurable": {"thread_id": "session_001"}}

# Uncomment to run interactively in a terminal
while True:
    user_input = input("Type here: ")
    if user_input.strip().lower() in ["exit", "quit", "bye"]:
        break
    initial_state = {"messages": [HumanMessage(content=user_input)]}
    print(f"User: {user_input}")
    # 3. Invoke with both State and Config
    response_state = chatbot_with_memory.invoke(initial_state, config=config)
    ai_message = response_state["messages"][-1].content
    print(f"AI: {ai_message}")